# CAN Bus IDS — Data Preprocessing & Feature Engineering Pipeline

This notebook implements **Stage 1 (Data Preprocessing and Feature Engineering)** of the
proposed framework: transforming raw CAN messages (Timestamp, CAN ID, DLC, Payload) into a
13-dimensional feature vector organized into 4 complementary categories:

| Category | Features | Count |
|---|---|---|
| Temporal | ΔT Global, ΔT Same-ID | 2 |
| Identifier-Based | CAN ID, CAN ID Entropy | 2 |
| Statistical | Payload Change Rate | 1 |
| Numerical Payload | Byte1 – Byte8 | 8 |

Before feature extraction, three preprocessing steps are performed sequentially:
1. **Timestamp Ordering** — sort all messages in ascending chronological order.
2. **Hexadecimal → Integer Conversion** — convert CAN ID and payload bytes from hex to decimal integers.
3. **Label Encoding** — map string class labels (AttackFree, DoS, Fuzzy, ...) to integers.

> Note: The code is written generically so it works with either dataset (Car-Hacking Dataset or
> CAN-MIRGU) — just adjust the column settings in the `CONFIG` cell below to match your file.


In [12]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from pathlib import Path

pd.set_option('display.max_columns', 50)


## 0. Configuration (CONFIG)

Update these values to match the actual column names in your dataset file.

- If your dataset follows the original Car-Hacking Dataset format (Hacking and Countermeasure
  Research Lab), typical columns look like: `Timestamp, CAN_ID, DLC, DATA0..DATA7, Flag/Label`.
- If your file is structured differently, just edit the dictionary below and the rest of the
  notebook will work as-is.


In [29]:
CONFIG = {
    # Path to the dataset file (csv). Change this to your actual path.
    "input_path": r"C:\Users\elmohandes\Desktop\Graduation project\Datasets\dataset_formatted.csv",

    # Name of the timestamp column (POSIX timestamp or any numeric timestamp)
    "timestamp_col": "timestamp",

    # Name of the CAN identifier column - whether stored as hex (string) or already numeric
    "can_id_col": "can_id",

    # Name of the DLC column (number of actual data bytes, 0-8)
    "dlc_col": "DLC",

    # Names of the 8 payload byte columns (if fewer than 8, they'll be NaN/missing and zero-padded)
    "payload_cols": [ "byte1", "byte2", "byte3", "byte4", "byte5", "byte6", "byte7","byte8"],

    # Name of the label column - contains normal / doS / fuzz  / ... etc.
    "label_col": "label",

    # Are CAN ID and payload byte values stored in hex (string) format, or already numeric?
    "hex_encoded": True,

    # Sliding window size used to compute CAN ID Entropy and Payload Change Rate
    "window_size": 50,

    # Desired class order for Label Encoding (0, 1, 2, ...)
    # Must match the ordering used elsewhere in your thesis
    "label_order": ["normal", "dos", "fuzz"],
}


## 1. Load the Dataset

In [14]:
import pandas as pd
df = pd.read_excel(CONFIG["input_path"])
df.head()
 


,timestamp,can_id,DLC,byte1,byte2,byte3,byte4,byte5,byte6,byte7,byte8,time_diff,label
0,1.781886e+09,166,4,208,50,0,24,0,0,0,0,0.000000,normal
1,1.781886e+09,158,8,0,0,0,0,0,0,0,25,0.000042,normal
2,1.781886e+09,161,8,0,0,5,80,1,8,0,28,0.000012,normal
3,1.781886e+09,00E,7,55,209,76,91,34,208,149,0,0.000028,normal
4,1.781886e+09,263,64,34,191,38,182,62,242,90,43,0.000610,fuzz


## 2. Preprocessing

### 2.1 Timestamp Ordering
Sort all messages in ascending chronological order to preserve the temporal sequence required
for time-difference computation.


In [17]:
df = df.sort_values(by=CONFIG["timestamp_col"], kind="mergesort").reset_index(drop=True)
df.head()


,timestamp,can_id,DLC,byte1,byte2,byte3,byte4,byte5,byte6,byte7,byte8,time_diff,label
0,1.781886e+09,166,4,208,50,0,24,0,0,0,0,0.000000,normal
1,1.781886e+09,158,8,0,0,0,0,0,0,0,25,0.000042,normal
2,1.781886e+09,161,8,0,0,5,80,1,8,0,28,0.000012,normal
3,1.781886e+09,00E,7,55,209,76,91,34,208,149,0,0.000028,normal
4,1.781886e+09,263,64,34,191,38,182,62,242,90,43,0.000610,fuzz


### 2.2 Hexadecimal → Integer Conversion
Convert CAN ID and payload bytes from hexadecimal notation into decimal integer values suitable
for machine learning algorithms.


In [18]:
def hex_to_int(val):
    """Converts a hex value (string) to an integer. Returns NaN if the value is empty/missing."""
    if pd.isna(val):
        return np.nan
    if isinstance(val, (int, np.integer, float, np.floating)):
        return int(val)
    s = str(val).strip()
    if s == "":
        return np.nan
    # strip 0x prefix if present
    s = s.replace("0x", "").replace("0X", "")
    try:
        return int(s, 16)
    except ValueError:
        return np.nan

if CONFIG["hex_encoded"]:
    df[CONFIG["can_id_col"]] = df[CONFIG["can_id_col"]].apply(hex_to_int)
    for col in CONFIG["payload_cols"]:
        if col in df.columns:
            df[col] = df[col].apply(hex_to_int)

# Any missing byte (DLC < 8) is zero-padded
for col in CONFIG["payload_cols"]:
    if col in df.columns:
        df[col] = df[col].fillna(0).astype(int)
    else:
        df[col] = 0

df[CONFIG["can_id_col"]] = df[CONFIG["can_id_col"]].astype(int)
df.head()


,timestamp,can_id,DLC,byte1,byte2,byte3,byte4,byte5,byte6,byte7,byte8,time_diff,label
0,1.781886e+09,166,4,208,50,0,24,0,0,0,0,0.000000,normal
1,1.781886e+09,158,8,0,0,0,0,0,0,0,25,0.000042,normal
2,1.781886e+09,161,8,0,0,5,80,1,8,0,28,0.000012,normal
3,1.781886e+09,14,7,55,209,76,91,34,208,149,0,0.000028,normal
4,1.781886e+09,263,64,34,191,38,182,62,242,90,43,0.000610,fuzz


### 2.3 Label Encoding
Map string attack labels (AttackFree, DoS, Fuzzy, ...) to integer values (0, 1, 2, ...).


In [31]:
le = LabelEncoder()
le.fit(CONFIG["label_order"])
df["label_encoded"] = le.transform(df[CONFIG["label_col"]])

 


## 3. Feature Engineering (13-Dimensional Feature Vector)

### Category 1 — Temporal Features

**ΔT Global**: time elapsed between any two consecutively observed CAN frames, regardless of ID.

**ΔT Same-ID**: time elapsed between consecutive frames sharing the same CAN identifier.


In [32]:
ts = CONFIG["timestamp_col"]
id_col = CONFIG["can_id_col"]

# Delta T Global
df["delta_t_global"] = df[ts].diff()
df["delta_t_global"] = df["delta_t_global"].fillna(0)

# Delta T Same-ID: difference between the current message and the last prior message with the same ID
df["delta_t_same_id"] = df.groupby(id_col)[ts].diff()
df["delta_t_same_id"] = df["delta_t_same_id"].fillna(0)

df[["delta_t_global", "delta_t_same_id"]].describe()


,delta_t_global,delta_t_same_id
count,495439.000000,495439.000000
mean,0.000218,0.311940
std,0.000718,1.059309
min,0.000000,0.000000
25%,0.000016,0.000289
50%,0.000095,0.009478
75%,0.000229,0.019879
max,0.018964,27.498378


### Category 2 — Identifier-Based Features

**CAN ID**: the decimal identifier value (already available after hex conversion).

**CAN ID Entropy**: Shannon entropy of the CAN ID distribution within a sliding window of size W,
measuring how random/regular the identifier stream is.


In [33]:
def rolling_shannon_entropy(id_series, window):
    """Computes the Shannon entropy of the CAN ID distribution within a sliding window of size `window`."""
    ids = id_series.to_numpy()
    n = len(ids)
    entropy = np.zeros(n)
    W = window

    for i in range(n):
        start = max(0, i - W + 1)
        window_vals = ids[start:i + 1]
        _, counts = np.unique(window_vals, return_counts=True)
        p = counts / counts.sum()
        entropy[i] = -np.sum(p * np.log2(p))

    return entropy

W = CONFIG["window_size"]
df["can_id_entropy"] = rolling_shannon_entropy(df[id_col], W)

df[[id_col, "can_id_entropy"]].describe()


,can_id,can_id_entropy
count,495439.000000,495439.000000
mean,262.610673,3.666683
std,390.386939,0.953871
min,0.000000,-0.000000
25%,0.000000,3.076217
50%,149.000000,3.523018
75%,380.000000,4.547976
max,2047.000000,5.643856


### Category 3 — Statistical Feature

**Payload Change Rate (PCR)**: the fraction of consecutive frames (sharing the same CAN ID) whose
payload differs from the preceding one, computed over a sliding window per identifier.


In [41]:
payload_cols = [c for c in CONFIG["payload_cols"] if c in df.columns]

def rolling_payload_change_rate(df, id_col, payload_cols, window):
    """
    For each message, computes the fraction of payload changes relative to the last `window`
    prior messages sharing the same CAN ID.
    """
    payload_matrix = df[payload_cols].to_numpy()
    ids = df[id_col].to_numpy()
    n = len(df)
    pcr = np.zeros(n)

    # For each ID, keep a list of indices in order of appearance
    from collections import defaultdict
    id_history = defaultdict(list)

    for i in range(n):
        cur_id = ids[i]
        hist = id_history[cur_id]
        if len(hist) == 0:
            pcr[i] = 0.0
        else:
            window_idx = hist[-window:]
            changes = 0
            prev_payload = payload_matrix[window_idx[0]]
            for idx in window_idx[1:]:
                cur_payload = payload_matrix[idx]
                if not np.array_equal(cur_payload, prev_payload):
                    changes += 1
                prev_payload = cur_payload
            # compare the last historical message against the current message
            if not np.array_equal(payload_matrix[i], prev_payload):
                changes += 1
            denom = len(window_idx)
            pcr[i] = changes / denom if denom > 0 else 0.0

        id_history[cur_id].append(i)

    return pcr

df["payload_change_rate"] = rolling_payload_change_rate(df, id_col, payload_cols, W)
 


### Category 4 — Numerical Payload Features

**Byte1 – Byte8**: raw byte values (0–255) after hex conversion and zero-padding for frames with
DLC less than 8. These columns are already available from the preprocessing step.


In [35]:
byte_rename = {orig: f"Byte{i+1}" for i, orig in enumerate(payload_cols)}
df = df.rename(columns=byte_rename)
byte_cols = list(byte_rename.values())
df[byte_cols].describe()


,Byte1,Byte2,Byte3,Byte4,Byte5,Byte6,Byte7,Byte8
count,495439.000000,495439.000000,495439.000000,495439.000000,495439.000000,495439.000000,495439.000000,495439.000000
mean,86.242942,80.892235,81.117807,89.811299,83.392141,72.087452,73.728398,77.599158
std,83.926865,84.506348,84.118303,85.476005,83.557507,83.337067,82.435144,79.044161
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,5.000000,0.000000,0.000000,0.000000,7.000000
50%,73.000000,50.000000,55.000000,69.000000,64.000000,31.000000,39.000000,48.000000
75%,154.000000,150.000000,153.000000,161.000000,160.000000,140.000000,139.000000,140.000000
max,255.000000,255.000000,255.000000,255.000000,255.000000,255.000000,255.000000,255.000000


## 4. Assemble the Final Feature Vector (13 dimensions)

In [40]:
feature_cols = [
    "delta_t_global", "delta_t_same_id",     # Temporal (2)
    id_col, "can_id_entropy",                # Identifier-Based (2)
    "payload_change_rate",                   # Statistical (1)
] + byte_cols                                 # Numerical Payload (8)

assert len(feature_cols) == 13, f"Expected 13 features, got {len(feature_cols)}"

final_df = df[feature_cols + ["label_encoded"]].copy()
final_df = final_df.rename(columns={id_col: "CAN_ID"})

final_df.head()


,delta_t_global,delta_t_same_id,CAN_ID,can_id_entropy,payload_change_rate,Byte1,Byte2,Byte3,Byte4,Byte5,Byte6,Byte7,Byte8,label_encoded
0,0.000000,0.0,166,-0.000000,0.0,208,50,0,24,0,0,0,0,2
1,0.000042,0.0,158,1.000000,0.0,0,0,0,0,0,0,0,25,2
2,0.000012,0.0,161,1.584963,0.0,0,0,5,80,1,8,0,28,2
3,0.000028,0.0,14,2.000000,0.0,55,209,76,91,34,208,149,0,2
4,0.000610,0.0,263,2.321928,0.0,34,191,38,182,62,242,90,43,1


## 5. Save the Processed Feature Dataset

In [39]:
output_path = Path("extracted feature.csv")
final_df.to_csv(output_path, index=False)
print(f"Saved to: {output_path.resolve()}")


Saved to: C:\Users\elmohandes\Desktop\Graduation project\extracted feature.csv
